In [ ]:
!nvidia-smi

In [ ]:
import os
# NOTE!
# this is for a multi-gpu setup. we basically set which GPU is visible for the program.
# if you only have one GPU (most likely the case ;) ), make sure this is set to "0"!
# or remove this line completely!
# otherwise you may accidentally make your GPU "invisible" to the program!
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# this is important on our server, as the default /tmp directory is very small.
# so if you compile lots of models, it will eventually fill up.
# if you are working on your own machine, you might want/have to remove this!
if os.path.isdir("/var/tmp"):
    os.environ["TMPDIR"] = "/var/tmp"
    os.environ["TORCHINDUCTOR_CACHE_DIR"] = "/var/tmp"

# Generative Adversarial Network

In [ ]:
import numpy as np
import torch
from matplotlib import pyplot as plt

from lgm.data import ImageDataModule
from lgm.models.gan import GANTrainingModuleParallel, optimization_inference
from lgm.ops import sum_except
from lgm.utils import interpolate
from lgm.visualization import ImagePlotter

torch.set_float32_matmul_precision('high')
torch.backends.cudnn.benchmark = True

## Part 1: Preamble

### Architecture Considerations
You can view D similarly to the encoder in a VAE, and G similar to the decoder.
This makes it relatively easy to build neural network architectures.
A few things to keep in mind:
- D should end with a _single output_ for binary classification.
Since we usually use logit-based cross-entropy losses, there should be no output activation.
- [Due to reasons](https://ovgu-ailab.github.io/blog/methods/2022/07/07/batchnorm-gans.html), D should not use batch normalization.
We can use `nn.GroupNorm` instead.
Batchnorm is okay for G, but you could also use Groupnorm there if you prefer.
- In principle, G can have an output activation that forces generations into the correct, range, such as [0, 1] for images.
Thus, a sigmoid activation on G could make sense.
However, this is currently not supported by our repository.
It would be easy to add, but you can also just use no output activation, forcing G to handle the correct output range by itself.
This seems to work well enough in practice.
- We once again need a _prior distribution_ for the latent space.
But other than for VAEs, we never need to backpropagate through the latents.
This means you are technically free to choose any distribution at all -- even discrete ones!
But a sensible default could be a standard normal distribution once again.

### Tips & Tricks
As discussed in the assignment text, there are many "little tricks" you can try on top of the standard GAN setup.
Most of these are relatively simple to use, and the more complicated ones are purposefully already supported by our code.

#### Optimizer improvements
Many GAN training loops make some changes to the standard optimizer setup:
- Reduce the learning rate.
Adam has a default set to `0.001`.
Try reducing this by a factor of 5-10, e.g. `0.0002`.
- It seems useful to reduce or completely remove momentum.
Intuitively, this might be because of the "moving target" we have in GAN training -- lower momentum allow each network to more quickly adapt to changes in the other one.
Adam uses a momentum of 0.9 per default; you could try 0.5 or even less.
Note that momentum is the first of the two numbers in the `betas` parameter.
- A little bit of weight decay might help with stability.
  However, you also risk weaking the model capacity, which could worsen the final results.

#### Rescaling data
It is often advocated to scale data such that it lies in the range [-1, 1] rathern than [0, 1].
This is trivially implemented by passing `normalize="center"` to our `ImageDataModule`.
An example is given in the starter notebook.
Note that we need to revert this transformation whenever plotting data, as functions like `imshow` usually expect image data in the [0, 1] range, but this is already handled by our code.
But if you write any custom plotting code, you may need to keep this in mind.

#### One-sided label smoothing
To prevent the D loss from saturating too much, we can employ _label smoothing_, replacing the 0/1 labels by soft versions.
[Improved Techniques for Training GANs](https://arxiv.org/pdf/1606.03498) proposes to only smooth the _positive_ labels away from 1 (e.g. 0.9), but leave the negative labels as 0.
We provide this as an option in the `GANTrainingModule`.
Note that this only makes sense for the standard cross-entropy loss, as this reduces saturation of the loss as outputs get close to 0/1.
If you use LSGAN (which we recommend -- see below), the exact value of the labels make no real difference.

#### Adding noise
We can add small amounts of noise to the data, as well as generated outputs, before putting them into D.
This makes the job slightly harder for the latter.
This is currently hardcoded in the training module via the `dequantizer` attribute.
You can experiment with turning this off.
To be honest, the impact of this wasn't fully tested.

#### EMA
A popular technique for modern generative models is to use an _exponential moving average (EMA)_ over model parameters.
During training, we keep an average of the parameters at each step.
After training, the parameters of the final learning step are replaced by the EMA parameters.
This supposed to smoothen the variance introduced by singular training steps, and can reduce issues with models "going in circles".

We provide `lgm.callbacks.EMAModel` for this.
You can add this to the callbacks in the config file.
The EMA model is not stored as a lightning checkpoint, but only as a standard torch `state_dict`.
You can load the weights like this:

```python
ema_path = f"lightning_logs/gan/{dataset}/{name}/version_{version}/checkpoints/ema_weights.pt"
lightning_module.load_state_dict(torch.load(ema_path))
```

This requires _first_ restoring the lightning module from the checkpoint!
You may need to adapt the path depending on your setup, but the `ema_weights.pt` should be in the same folder as your model checkpoints.

#### LSGAN
There are other possibilities to use for the loss function besides the
classical cross-entropy.
One is the Least-Squares-GAN (LSGAN).
While in principle there is quite a bit of theory as well as different possible setups, the basic idea is just to use the squared error for the D loss.
That's it!
You can simply pass `d_loss_fn=nn.MSELoss()` to the training module to implement this.

#### Batch features
We can help D enforce variety in the generated batches by explicitly computing features _over the batch_, rather than the typical per-input features.
A simple example is the `SDLayer` that we added to the repository, which was originally proposed for [Progressive Growing GAN](https://arxiv.org/pdf/1710.10196).
This can easily be added by a single `sd_layer` argument in the `Discriminator` class.

#### Feature Matching
This is likely the most important of all these techniques.
The intuition is quite simple.
If we had a good G that was producing samples whose overall distribution looks like the data distribution, then the features computed by D in the hidden layers should _on average_ be similar between the real data and samples from G.
This means:
- Compute features for real data and average.
- Compute features for generated data and average.
- Compute the squared difference between these averages.
Note: NOT the average difference -- rather the difference of averages!
- Since this difference should be small, we can use it as a loss function for G!

In principle, we can just add this so-called _feature matching loss_ to the regular G loss, or even replace the old one completely.
This is a very powerful technique that most modern GANs will employ in some way.

There is just one problem:
How do we get the features?
Turns out we must adapt our D model to return _hidden layer outputs_ (i.e. "features") in addition to its regular classification output.
This can be achieved by a change to our `lgm.layers` module.
Your discriminator _must_ use `return_all=True` in its `config` argument, as the code is set up to always expect these additional outputs, even when feature matching is turned off (they are just ignored in that case).

Which features to use is completely arbitray.
We decide to return all _block outputs_.
Then the difference outlined above is computed for each block output, and everything is summed together.
This ensures that both early, mid and late features from D give a training signal to G.

Note with this setup it's quite likely that D achieves pretty much perfect classification, and yet G is still learning fine.
We don't _have to_ balance the "game" between the two networks, we just have to make sure there is a good training signal for G, and this is much simpler with feature matching.
Try it out!
Note that the `feature_matching` argument for the training module should be a float, giving the relative weight of the feature loss compared to the standard classification loss for G.

#### Model Size
A common problem with basic GANs is D "overpowering" G.
In our setup, we can supply different architectures for both networks.
As a reminder, we have videos ([LINK](https://cloud.ovgu.de/s/BaDbzLBQRgwE7yY)) explaining the CNN structures we use here, since we likely don't have time to cover this in detail in the exercise itself.
We got decent results by using a smaller network for D, cutting off the last level, which has the most channels and thus also the most parameters.

#### Other Model Components
One popular method for stabilizing GANs is [Spectral Normalization](https://arxiv.org/pdf/1802.05957).
This essentially restricts how much D can change its output when the input changes by a certain amount, acting as a very strong regularizer.
Torch [already implements it](https://docs.pytorch.org/docs/2.15/generated/torch.nn.utils.parametrizations.spectral_norm.html#torch.nn.utils.parametrizations.spectral_norm), but actually using it would require changing our model code to wrap every single weight layer with this function, which is why we have not included this.
But in principle, you could change the `layers` code accordingly; it would just be quite a bit of work.
You would also have to remove normalization (i.e. use `"dummy"` norm).

[StyleGAN 2](https://arxiv.org/pdf/1912.04958) proposes variants on the standard feed-forward CNN architectures.
You can find these in the paper on page 6, section 4.1 and figure 7.
We have implemented this in the repository:
Input/output skips can be used by activating the corresponding fields in `EncoderConfig`/`DecoderConfig`.
The residual configurations can be used by activating `level_residual` in the `BodyConfig` that both encoder/decoder use.
However, don't expect too much of a difference for low-resolution images.

## Part 2: Reloading

In [ ]:
dataset = "fashion"
# baseline version_0 stopped at epoch 16. The finished run is version_1.
runs = [
    ("baseline", 1),
    ("lsgan_adam", 0),
    ("featmatch_smalld", 0),
    ("full", 0),
]

check_paths = {
    name: f"lightning_logs/gan/{dataset}/{name}/version_{version}/checkpoints/last.ckpt"
    for name, version in runs
}

data_module = ImageDataModule.load_from_checkpoint(check_paths["baseline"])
data_module.prepare_data()
data_module.setup()
plotter = ImagePlotter(transform=data_module.normalizer.invert)


In [ ]:
lightning_modules = {}
models = {}
for name, version in runs:
    lightning_module = GANTrainingModuleParallel.load_from_checkpoint(check_paths[name])
    ema_path = f"lightning_logs/gan/{dataset}/{name}/version_{version}/checkpoints/ema_weights.pt"
    if os.path.isfile(ema_path):
        lightning_module.load_state_dict(torch.load(ema_path, map_location="cpu", weights_only=True))
    model = lightning_module.model.eval()
    model.compile()
    lightning_modules[name] = lightning_module
    models[name] = model
    print(name, "loaded")
device = next(iter(lightning_modules.values())).device
print("Device is", device)


In [ ]:
for name, model in models.items():
    print("=" * 60)
    print(name)
    print(model)


## Part 3: Exploration

### Latent Space

Just as with VAEs, we can interpolate between images in the latent space, achieving a "morph" effect. We just can't _encode_ images (or can we? see below...), so we make do with randomly generated ones.

In [ ]:
n_rows = 10
for name, model in models.items():
    with torch.inference_mode():
        generations = model.generate(n_rows**2)
    _ = plotter.image_grid(generations,
                           title=f"Generation Examples — {name}",
                           n_rows=n_rows)


In [ ]:
codes_by_model = {}
for name, model in models.items():
    codes = model.generator.sample_noise(2)
    codes_by_model[name] = codes
    code1, code2 = codes

    images = model.generator(codes)
    image1, image2 = images

    # see the images we are dealing with here
    _ = plotter.image_grid(images,
                           title=name,
                           n_rows=1,
                           n_cols=2)


In [ ]:
n_rows = 8
interpolated_by_model = {}
for name, model in models.items():
    code1, code2 = codes_by_model[name]
    interpolated_codes = interpolate(code1, code2, n_interpolations=n_rows**2)

    with torch.inference_mode():
        interpolated_images = model.generator(interpolated_codes)
    interpolated_by_model[name] = interpolated_images
    _ = plotter.image_grid(interpolated_images,
                           title=f"Latent Interpolations — {name}",
                           n_rows=n_rows, suppress_plots=True)


In [ ]:
for name, interpolated_images in interpolated_by_model.items():
    plotter.make_animation(f"animations/interpolate_gan_{dataset}_{name}",
                           torch.cat((interpolated_images,
                                      torch.flip(interpolated_images[1:-1], dims=[0]))),
                           invert_colors=True)


### GAN Inference Through Optimization

Doctors said that GANs couldn't do inference.
This man didn't believe them.
This is him, 1000 optimization steps later:

...Anyway, what we are doing here is:
- Start with randomly initialized noise.
- Send the noise through the GAN to get an image output.
- Compare this to the desired image we want to "encode" via some difference function.
- Backpropagate into the noise and adapt it such that the output will be closer to the desired target.
- Repeat the three steps above until convergence.

Thus, we can do "inference" by finding a noise vector that closely correspond to a desired target image.
However, there is no guarantee that the GAN can really match the target (but then again, neither do VAE reconstructions match the inputs).

There is also no guarantee that the optimized noise vector will correspond to the prior we used to train the model.
That is why we add a regularizer that encourages the vector to have a norm similar to what would be expected of Standard Normal samples ($\sqrt{d}$ where $d$ is the dimensionality of the noise space).

In [ ]:
target_images = next(iter(data_module.test_dataloader()))[0].to(device)
optimized_by_model = {}
for name, model in models.items():
    optimized_by_model[name] = optimization_inference(target_images,
                                                      model.generator,
                                                      n_steps=5000,
                                                      plotter=plotter,
                                                      regularization=0.0)


We can check the norms of the generated noise compared to what we would expect from actual noise function outputs.
This gives us an idea of how well our regularization worked, and if we may have to increase it.
Note, this will usually reduce the chance of finding good generations.

In [ ]:
for name, model in models.items():
    optimized_noise = optimized_by_model[name]
    random_noise = model.generator.sample_noise(target_images.shape[0]).cpu()
    random_norms = torch.sqrt(sum_except(random_noise**2))
    plt.hist(random_norms, bins=20)
    plt.title(f"Ideal noise norms — {name}")
    plt.show()

    optimized_norms = torch.sqrt(sum_except(optimized_noise**2)).detach().cpu()
    plt.hist(optimized_norms, bins=20)
    plt.title(f"Optimized noise norms — {name}")
    plt.show()
